
# Notebook creativo: análisis con `dataspark.statistical`

Objetivo: mostrar un flujo práctico del módulo estadístico con un toque creativo.

Flujo:
1. Cargar datos.
2. Limpiar **rápidamente** con `dataspark.cleansing`.
3. Ejecutar pruebas paramétricas y no paramétricas.
4. Cuantificar magnitud de efecto y potencia.



## Dataset

Usamos Titanic desde URL pública:
- `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv`

Fallback local en caso de red restringida: `sklearn.load_breast_cancer`.


In [ ]:

import numpy as np
import pandas as pd

from dataspark.cleansing import DataCleaner
from dataspark.statistical import HypothesisTester, NonParametricTests, EffectSizeCalculator

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [ ]:

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

try:
    raw_df = pd.read_csv(url)
    source_name = f"Titanic (URL): {url}"
except Exception as exc:
    from sklearn.datasets import load_breast_cancer
    bc = load_breast_cancer(as_frame=True)
    raw_df = bc.frame
    source_name = "Fallback local: sklearn.datasets.load_breast_cancer()"
    print("No se pudo descargar dataset remoto:", exc)

print("Fuente:", source_name)
print("Shape original:", raw_df.shape)
raw_df.head()


## 1) Limpieza breve con `DataCleaner` (sin extendernos)


In [ ]:

df = raw_df.copy()

# ruido mínimo para demostrar limpieza
if "fare" in df.columns:
    df = df.rename(columns={"fare": " Fare "})

rng = np.random.default_rng(123)
if "age" in df.columns:
    idx = rng.choice(len(df), size=min(20, len(df)), replace=False)
    df.loc[idx, "age"] = np.nan

cleaner = DataCleaner(missing_strategy="median", standardize_columns=True, strip_whitespace=True)
df_clean = cleaner.fit_transform(df)

print("Shape limpio:", df_clean.shape)
print("Nulos totales:", int(df_clean.isna().sum().sum()))
df_clean.head()



## 2) Preparación de grupos para pruebas

Para hacer el análisis más interesante:
- comparamos tarifas entre sobrevivientes y no sobrevivientes,
- comparamos edades entre clases de pasajero,
- construimos tabla de contingencia para `sex` vs `survived`.


In [ ]:

if "survived" in df_clean.columns:
    g0 = df_clean.loc[df_clean["survived"] == 0, "fare" if "fare" in df_clean.columns else df_clean.select_dtypes(include="number").columns[0]].dropna()
    g1 = df_clean.loc[df_clean["survived"] == 1, "fare" if "fare" in df_clean.columns else df_clean.select_dtypes(include="number").columns[0]].dropna()
else:
    num = df_clean.select_dtypes(include="number").columns.tolist()
    g0 = df_clean[num[0]].dropna().sample(frac=0.5, random_state=1)
    g1 = df_clean[num[0]].dropna().drop(g0.index, errors="ignore")

print("n grupo 0:", len(g0), "| n grupo 1:", len(g1))


## 3) `HypothesisTester`: pruebas paramétricas


In [ ]:

# t-test independiente (Welch por defecto)
t_ind = HypothesisTester.t_test(g0, g1, alternative="two-sided", equal_var=False)
t_ind


In [ ]:

# t-test pareado (ejemplo sintético sencillo)
paired_before = np.array([10, 12, 13, 11, 14, 15, 13, 12], dtype=float)
paired_after = paired_before + np.array([1, 0, 2, 1, 1, 2, 0, 1], dtype=float)

paired = HypothesisTester.paired_t_test(paired_before, paired_after)
paired


In [ ]:

# ANOVA: comparar "age" por clase (si existe)
if "class" in df_clean.columns and "age" in df_clean.columns:
    groups = [g["age"].dropna().values for _, g in df_clean.groupby("class") if g["age"].notna().sum() > 2]
elif "pclass" in df_clean.columns and "age" in df_clean.columns:
    groups = [g["age"].dropna().values for _, g in df_clean.groupby("pclass") if g["age"].notna().sum() > 2]
else:
    num = df_clean.select_dtypes(include="number").columns.tolist()
    x = df_clean[num[0]].dropna().values
    third = len(x) // 3
    groups = [x[:third], x[third:2*third], x[2*third:3*third]]

anova_res = HypothesisTester.one_way_anova(*groups)
anova_res


In [ ]:

# Chi-cuadrado: independencia sexo vs supervivencia (si aplica)
if set(["sex", "survived"]).issubset(df_clean.columns):
    contingency = pd.crosstab(df_clean["sex"], df_clean["survived"])
else:
    # fallback binning numérico
    num = df_clean.select_dtypes(include="number").columns.tolist()[0]
    contingency = pd.crosstab(pd.qcut(df_clean[num], q=2, duplicates="drop"), df_clean[df_clean.select_dtypes(include="number").columns[1] > df_clean[df_clean.select_dtypes(include="number").columns[1]].median())

chi2_res = HypothesisTester.chi_squared(contingency)
chi2_res


In [ ]:

# Z-test de proporciones
if set(["sex", "survived"]).issubset(df_clean.columns):
    male = df_clean[df_clean["sex"] == "male"]
    female = df_clean[df_clean["sex"] == "female"]
    succ_a, n_a = int(male["survived"].sum()), len(male)
    succ_b, n_b = int(female["survived"].sum()), len(female)
else:
    succ_a, n_a = 45, 120
    succ_b, n_b = 70, 130

prop_z = HypothesisTester.proportion_z_test(succ_a, n_a, succ_b, n_b)
prop_z


## 4) `NonParametricTests`: alternativas robustas


In [ ]:

mw = NonParametricTests.mann_whitney(g0, g1)
ks = NonParametricTests.ks_two_sample(g0, g1)

mw, ks


In [ ]:

# Wilcoxon (sobre vectores pareados del ejemplo)
wil = NonParametricTests.wilcoxon_signed_rank(paired_before, paired_after)
wil


In [ ]:

# Kruskal/Friedman según grupos disponibles
krus = NonParametricTests.kruskal_wallis(*groups)

# Friedman requiere igual longitud; hacemos recorte común
min_len = min(len(g) for g in groups)
if min_len >= 3 and len(groups) >= 3:
    fried = NonParametricTests.friedman(*[g[:min_len] for g in groups[:3]])
else:
    fried = {"note": "No hay grupos suficientes/equiparables para Friedman"}

krus, fried


In [ ]:

# Runs test sobre una serie numérica
series_for_runs = df_clean.select_dtypes(include="number").iloc[:, 0].dropna()
runs = NonParametricTests.runs_test(series_for_runs)
runs


## 5) `EffectSizeCalculator`: magnitud del efecto y potencia


In [ ]:

cohen = EffectSizeCalculator.cohens_d(g0, g1)
cramer = EffectSizeCalculator.cramers_v(contingency)
eta = EffectSizeCalculator.eta_squared(*groups)

cohen, cramer, eta


In [ ]:

# Power analysis: (a) potencia lograda dado n, (b) n requerido dada potencia objetivo
power_given_n = EffectSizeCalculator.power_analysis(effect_size=0.4, n=80, alpha=0.05)
n_for_power = EffectSizeCalculator.power_analysis(effect_size=0.4, power=0.8, alpha=0.05)

power_given_n, n_for_power



## 6) Lectura rápida de resultados

- Usa p-values para evidencia estadística (significancia).
- Usa tamaños de efecto para relevancia práctica.
- Usa potencia para planificar tamaños de muestra más confiables.

Con este patrón puedes pasar de limpieza mínima a inferencia estadística robusta.
